In [ ]:
import os
import io  # <--- ĐÃ THÊM
from flask import send_file  # <--- ĐÃ THÊM
import tensorflow as tf
from flask import Flask, request, jsonify, render_template, send_from_directory
import librosa
import numpy as np
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Tắt cảnh báo
warnings.filterwarnings('ignore')

# === 1. Tải mô hình và các cài đặt ===
print(">>> Đang tải mô hình Keras... (việc này mất vài giây)")
MODEL_PATH = r'D:\Python_mohinh\MODEL_CHUAN_LAN_NAY.keras'
model = tf.keras.models.load_model(MODEL_PATH)
print(f">>> Tải mô hình từ {MODEL_PATH} thành công!")

# Các thiết lập phải GIỐNG HỆT lúc train
class_names = ['Electronic', 'Experimental', 'Folk', 'Hip-Hop',
               'Instrumental', 'International', 'Pop', 'Rock']
IMG_HEIGHT = 128
IMG_WIDTH = 860
DURATION = 20

# === 2. Các hàm xử lý âm thanh (Chuẩn 3 Kênh) ===
def read_audio_for_prediction(pathname):
    y, sr = librosa.load(pathname, mono=True, sr=22050, duration=DURATION)
    samples_needed = 22050 * DURATION
    if len(y) < samples_needed:
        padding = samples_needed - len(y)
        offset = padding // 2
        y = np.pad(y, (offset, samples_needed - len(y) - offset), 'constant')
    if len(y) > samples_needed:
        y = y[:samples_needed]
    return y

def audio_to_melspectrogram_for_prediction(audio):
    spectrogram = librosa.feature.melspectrogram(
        y=audio, sr=22050, n_mels=IMG_HEIGHT,
        hop_length=512, n_fft=2048,
        fmin=20, fmax=8000
    )
    spectrogram_db = librosa.power_to_db(spectrogram, ref=np.max)
    mels_resized = spectrogram_db[:, :IMG_WIDTH]
    mels_rgb = np.stack((mels_resized,) * 3, axis=-1)
    return mels_rgb.astype("float32")

def process_mp3_to_image(file_storage):
    try:
        temp_path = "temp_audio.mp3"
        file_storage.save(temp_path)
        audio = read_audio_for_prediction(temp_path)
        mels_rgb = audio_to_melspectrogram_for_prediction(audio)
        mels_batch = np.expand_dims(mels_rgb, axis=0)
        os.remove(temp_path)
        return mels_batch
    except Exception as e:
        print(f"Lỗi xử lý audio: {e}")
        return None

# === HÀM TẠO ẢNH MEL-SPECTROGRAM ===
def create_spectrogram_image(mels_rgb):
    try:
        # === CHUẨN HÓA VỀ [0, 1] ĐỂ TRÁNH CLIPPING ===
        mels_normalized = mels_rgb - mels_rgb.min()
        mels_normalized = mels_normalized / mels_normalized.max()

        fig = plt.figure(figsize=(8.6, 1.28), dpi=100)
        ax = fig.add_subplot(111)
        ax.imshow(mels_normalized, aspect='auto', origin='lower', cmap='magma')
        ax.axis('off')
        
        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0, transparent=False)
        buf.seek(0)
        plt.close(fig)
        return buf
    except Exception as e:
        print(f"Lỗi tạo ảnh: {e}")
        if 'fig' in locals():
            plt.close(fig)
        return None

# === 3. Flask App ===
app = Flask(__name__)

# --- ROUTE: GIAO DIỆN WEB ---
@app.route("/")
def index():
    return send_from_directory('.', 'index.html')

# --- ROUTE: FILE TĨNH (CSS, JS, VIDEO, v.v.) ---
@app.route('/<path:filename>')
def static_files(filename):
    return send_from_directory('.', filename)

# --- ROUTE: DỰ ĐOÁN THỂ LOẠI ---
@app.route("/predict", methods=['POST'])
def predict_genre():
    if 'file' not in request.files:
        return jsonify({"error": "Không tìm thấy file"}), 400
    file = request.files['file']
    if file.filename == '':
        return jsonify({"error": "Chưa chọn file"}), 400

    try:
        spectrogram_batch = process_mp3_to_image(file)
        if spectrogram_batch is None:
            return jsonify({"error": "Không thể xử lý file âm thanh"}), 500

        predictions = model.predict(spectrogram_batch)
        predicted_index = np.argmax(predictions[0])  # <--- ĐÃ SỬA: [00] → [0]
        predicted_genre = class_names[predicted_index]
        confidence = float(np.max(predictions[0]))

        return jsonify({
            "genre_du_doan": predicted_genre,
            "do_tin_cay": f"{confidence * 100:.2f}%",
            "all_scores": {class_names[i]: float(predictions[0][i]) for i in range(len(class_names))}
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# --- ROUTE MỚI: TRẢ VỀ ẢNH MEL-SPECTROGRAM ---
@app.route("/spectrogram", methods=['POST'])
def get_spectrogram():
    if 'file' not in request.files:
        return jsonify({"error": "Không tìm thấy file"}), 400
    file = request.files['file']
    if file.filename == '':
        return jsonify({"error": "Chưa chọn file"}), 400

    try:
        spectrogram_batch = process_mp3_to_image(file)
        if spectrogram_batch is None:
            return jsonify({"error": "Không thể xử lý file âm thanh"}), 500

        img_buf = create_spectrogram_image(spectrogram_batch[0])
        return send_file(img_buf, mimetype='image/png')

    except Exception as e:
        return jsonify({"error": str(e)}), 500

# === 4. Chạy server ===
if __name__ == "__main__":
    app.run(debug=False, use_reloader=False, port=5000, host='0.0.0.0')

>>> Đang tải mô hình Keras... (việc này mất vài giây)
>>> Tải mô hình từ D:\Python_mohinh\MODEL_CHUAN_LAN_NAY.keras thành công!
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.13:5000
Press CTRL+C to quit
127.0.0.1 - - [27/Oct/2025 02:42:08] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [27/Oct/2025 02:42:09] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:42:09] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:42:10] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:42:10] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:42:11] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:42:11] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:42:18] "GET /?zarsrc=31&utm_source=zalo&utm_medium=zalo&utm_campaign=zalo HTTP/1.1" 200 -
127.0.0.1 - - [27/Oct/2025 02:42:18] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:42:19] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:42:19] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oc

1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step


127.0.0.1 - - [27/Oct/2025 02:42:50] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [27/Oct/2025 02:42:52] "POST /spectrogram HTTP/1.1" 200 -
127.0.0.1 - - [27/Oct/2025 02:44:20] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:44:21] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:46:14] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:46:14] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:46:16] "GET / HTTP/1.1" 304 -
127.0.0.1 - - [27/Oct/2025 02:46:16] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:46:16] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:46:16] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:46:16] "GET /background.mp4 HTTP/1.1" 206 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 268ms/step


127.0.0.1 - - [27/Oct/2025 02:47:13] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [27/Oct/2025 02:47:21] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:47:21] "GET /background.mp4 HTTP/1.1" 206 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 401ms/step


127.0.0.1 - - [27/Oct/2025 02:47:33] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [27/Oct/2025 02:47:34] "POST /spectrogram HTTP/1.1" 200 -
127.0.0.1 - - [27/Oct/2025 02:51:03] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:53:52] "GET /background.mp4 HTTP/1.1" 206 -
127.0.0.1 - - [27/Oct/2025 02:53:52] "GET /background.mp4 HTTP/1.1" 206 -
